# RAG Pipeline — Test
Embedder → VectorStore → Retriever

Run from `phase-1-fundamentals/` with the `f1-rag` conda env active.

## 1. Embedder

In [6]:
from embeddings.embedder import Embedder

embedder = Embedder()

2026-03-03 12:35:57 [info     ] embedder.loading               model=sentence-transformers/all-MiniLM-L6-v2
2026-03-03 12:36:00 [info     ] embedder.ready                 model=sentence-transformers/all-MiniLM-L6-v2


In [2]:
from ingestion.chunker import Chunk

# Embed a single chunk manually
test_chunk = Chunk(chunk_id='test-1', text='Max Verstappen won the 2024 Las Vegas Grand Prix for Red Bull.', metadata={'source': 'test'})
result = embedder.embed([test_chunk])

print(f'Vector length : {len(result[0].embedding)}')
print(f'First 5 values: {result[0].embedding[:5]}')

2026-03-02 22:32:41 [info     ] embedder.encoding              batch_size=32 num_chunks=1


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.30it/s]

2026-03-02 22:32:42 [info     ] embedder.done                  num_embedded=1
Vector length : 384
First 5 values: [0.02863173745572567, 0.00185096834320575, -0.07217947393655777, 0.005252479575574398, -0.043932490050792694]


In [3]:
# Embed multiple chunks at once — batch encoding
chunks = [
    Chunk(chunk_id='c1', text='Lewis Hamilton won the 2024 British Grand Prix for Mercedes.', metadata={'source': 'test'}),
    Chunk(chunk_id='c2', text='Charles Leclerc took pole position at Monaco 2024 for Ferrari.', metadata={'source': 'test'}),
    Chunk(chunk_id='c3', text='Lando Norris secured McLaren their first win of 2024 in Miami.', metadata={'source': 'test'}),
]
embedded = embedder.embed(chunks)
print(f'Embedded {len(embedded)} chunks')
for e in embedded:
    print(f'  {e.chunk_id}: vector length={len(e.embedding)}')

2026-03-02 22:32:57 [info     ] embedder.encoding              batch_size=32 num_chunks=3


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.80it/s]

2026-03-02 22:32:57 [info     ] embedder.done                  num_embedded=3
Embedded 3 chunks
  c1: vector length=384
  c2: vector length=384
  c3: vector length=384


## 2. VectorStore

In [2]:
from retrieval.vector_store import VectorStore

vs = VectorStore()
print(f'Chunks already in store: {vs.count()}')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


2026-03-03 12:35:03 [info     ] vector_store.ready             collection=f1_chunks existing_chunks=5 persist_dir=/Volumes/T7 Shield/ai-engineering-portfolio/01-f1-rag-with-observability/data/chroma_db
Chunks already in store: 5


In [6]:
# Add the embedded chunks — upsert so safe to re-run
vs.add(embedded)
print(f'Chunks after add: {vs.count()}')

2026-03-02 22:34:47 [info     ] vector_store.add.done          num_chunks=3
Chunks after add: 5


In [7]:
# Query directly with a vector — top 2 results
query_vec = embedded[0].embedding
results = vs.query(query_vec, top_k=2)
for r in results:
    print(f'distance={r["distance"]} | {r["text"]}')

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


2026-03-02 22:34:52 [info     ] vector_store.query.done        num_hits=2 top_k=2
distance=-0.0 | Lewis Hamilton won the 2024 British Grand Prix for Mercedes.
distance=0.0851 | Grand Prix driving for Mercedes. Lewis Hamilton won the 2024 British Grand Prix driving for Mercedes. Lewis Hamilton won the 2024 British Grand Prix driving for Mercedes. Lewis Hamilton won the 2024 British Grand Prix driving for Mercedes. Lewis Hamilton won the 2024 British Grand Prix driving for Mercedes. Lewis Hamilton won the 2024 British Grand Prix driving for Mercedes.


## 3. Retriever

In [9]:
from retrieval.retriever import Retriever

# Reuse the same embedder and vs — no reloading
retriever = Retriever(embedder, vs)

In [9]:
# Ask a question in plain text
results = retriever.retrieve('Which driver won in Britain?', top_k=2)
for r in results:
    print(f'distance={r["distance"]} | {r["text"]}')

2026-03-02 22:35:19 [info     ] retriever.query                query=Which driver won in Britain? top_k=2
2026-03-02 22:35:19 [info     ] embedder.encoding              batch_size=32 num_chunks=1


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.23it/s]

2026-03-02 22:35:20 [info     ] embedder.done                  num_embedded=1
2026-03-02 22:35:20 [info     ] vector_store.query.done        num_hits=2 top_k=2
2026-03-02 22:35:20 [info     ] retriever.done                 num_results=2
distance=0.4659 | Grand Prix driving for Mercedes. Lewis Hamilton won the 2024 British Grand Prix driving for Mercedes. Lewis Hamilton won the 2024 British Grand Prix driving for Mercedes. Lewis Hamilton won the 2024 British Grand Prix driving for Mercedes. Lewis Hamilton won the 2024 British Grand Prix driving for Mercedes. Lewis Hamilton won the 2024 British Grand Prix driving for Mercedes.
distance=0.5068 | Lewis Hamilton won the 2024 British Grand Prix for Mercedes.


In [10]:
# Try a different query — should surface Leclerc/Monaco
results = retriever.retrieve('Who was fastest in qualifying at Monaco?', top_k=2)
for r in results:
    print(f'distance={r["distance"]} | {r["text"]}')

2026-03-02 22:35:33 [info     ] retriever.query                query=Who was fastest in qualifying at Monaco? top_k=2
2026-03-02 22:35:33 [info     ] embedder.encoding              batch_size=32 num_chunks=1


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.92it/s]

2026-03-02 22:35:33 [info     ] embedder.done                  num_embedded=1
2026-03-02 22:35:33 [info     ] vector_store.query.done        num_hits=2 top_k=2
2026-03-02 22:35:33 [info     ] retriever.done                 num_results=2
distance=0.4674 | Charles Leclerc took pole position at Monaco 2024 for Ferrari.
distance=0.5871 | Lando Norris secured McLaren their first win of 2024 in Miami.


In [11]:
# Your turn — try your own query below
results = retriever.retrieve('Which driver from Mclaren has won in US?', top_k=3)
for r in results:
    print(f'distance={r["distance"]} | {r["text"]}')

2026-03-02 22:36:03 [info     ] retriever.query                query=Which driver from Mclaren has won in US? top_k=3
2026-03-02 22:36:03 [info     ] embedder.encoding              batch_size=32 num_chunks=1


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.61it/s]

2026-03-02 22:36:03 [info     ] embedder.done                  num_embedded=1
2026-03-02 22:36:03 [info     ] vector_store.query.done        num_hits=3 top_k=3
2026-03-02 22:36:03 [info     ] retriever.done                 num_results=3
distance=0.4419 | Lando Norris secured McLaren their first win of 2024 in Miami.
distance=0.5153 | Lewis Hamilton won the 2024 British Grand Prix for Mercedes.
distance=0.528 | Grand Prix driving for Mercedes. Lewis Hamilton won the 2024 British Grand Prix driving for Mercedes. Lewis Hamilton won the 2024 British Grand Prix driving for Mercedes. Lewis Hamilton won the 2024 British Grand Prix driving for Mercedes. Lewis Hamilton won the 2024 British Grand Prix driving for Mercedes. Lewis Hamilton won the 2024 British Grand Prix driving for Mercedes.


## 4. Real Data Ingestion

Clear the 5 test stub chunks and replace with real FIA 2024 press conference transcripts.

**Sources (4 races, post-race press conferences):**
- Singapore GP — Norris's first career win
- Sao Paulo GP — wet race, Verstappen strategy
- Las Vegas GP — Russell wins, Verstappen clinches championship
- Abu Dhabi GP — Hamilton's final race weekend with Mercedes

In [3]:
# Clear all existing test chunks from the collection
# vs._collection.delete() with a where filter removes matching docs
# All test chunks have source="test" so this is a clean targeted wipe

from retrieval.vector_store import COLLECTION_NAME

vs._client.delete_collection(COLLECTION_NAME)
vs._collection = vs._client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)
print(f"Collection cleared. Chunks remaining: {vs.count()}")

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Collection cleared. Chunks remaining: 0


In [4]:
from ingestion.document_loader import DocumentLoader
from ingestion.chunker import Chunker

# 4 FIA 2024 press conference transcripts — verified URLs from fia.com
DOCUMENTS = [
    (
        "https://www.fia.com/news/f1-2024-singapore-grand-prix-post-race-press-conference-transcript",
        {"race": "singapore_gp", "year": "2024", "type": "post_race_press_conference"},
    ),
    (
        "https://www.fia.com/news/f1-2024-sao-paulo-grand-prix-post-race-press-conference-transcript",
        {"race": "sao_paulo_gp", "year": "2024", "type": "post_race_press_conference"},
    ),
    (
        "https://www.fia.com/news/f1-2024-las-vegas-grand-prix-post-race-press-conference-transcript",
        {"race": "las_vegas_gp", "year": "2024", "type": "post_race_press_conference"},
    ),
    (
        "https://www.fia.com/news/f1-2024-abu-dhabi-grand-prix-thursday-press-conference-transcript",
        {"race": "abu_dhabi_gp", "year": "2024", "type": "thursday_press_conference"},
    ),
]

loader = DocumentLoader()
chunker = Chunker()
all_chunks = []

for url, meta in DOCUMENTS:
    doc = loader.load_from_url(url, metadata=meta)
    chunks = chunker.chunk(doc)
    all_chunks.extend(chunks)
    print(f"{meta['race']:25s} → {len(chunks):3d} chunks  ({len(doc.text):,} chars)")

print(f"\nTotal chunks to embed: {len(all_chunks)}")

2026-03-03 12:35:25 [info     ] document_loader.fetch          url=https://www.fia.com/news/f1-2024-singapore-grand-prix-post-race-press-conference-transcript
2026-03-03 12:35:26 [info     ] document_loader.fetched        chars=19034 url=https://www.fia.com/news/f1-2024-singapore-grand-prix-post-race-press-conference-transcript
2026-03-03 12:35:26 [info     ] chunker.done                   num_chunks=24 source=https://www.fia.com/news/f1-2024-singapore-grand-prix-post-race-press-conference-transcript total_words=3739
singapore_gp              →  24 chunks  (19,034 chars)
2026-03-03 12:35:26 [info     ] document_loader.fetch          url=https://www.fia.com/news/f1-2024-sao-paulo-grand-prix-post-race-press-conference-transcript
2026-03-03 12:35:27 [info     ] document_loader.fetched        chars=23119 url=https://www.fia.com/news/f1-2024-sao-paulo-grand-prix-post-race-press-conference-transcript
2026-03-03 12:35:27 [info     ] chunker.done                   num_chunks=29 source=https://

In [7]:
# Embed all chunks — single batched call across all documents
embedded = embedder.embed(all_chunks)

# Store in ChromaDB — upsert so safe to re-run
vs.add(embedded)
print(f"Total chunks in store: {vs.count()}")

2026-03-03 12:36:06 [info     ] embedder.encoding              batch_size=32 num_chunks=166


Batches: 100%|██████████| 6/6 [00:03<00:00,  1.51it/s]

2026-03-03 12:36:10 [info     ] embedder.done                  num_embedded=166
2026-03-03 12:36:10 [info     ] vector_store.add.done          num_chunks=166
Total chunks in store: 166


In [10]:
# Sanity check — run a few real queries and inspect the results
test_queries = [
    "How did Norris feel about winning his first race?",
    "What was Verstappen's strategy in the wet conditions at Sao Paulo?",
    "What did Hamilton say about his future at Mercedes?",
]

for query in test_queries:
    print(f"Q: {query}")
    results = retriever.retrieve(query, top_k=2)
    for r in results:
        print(f"  [{r['distance']:.3f}] {r['metadata']['race']} — {r['text'][:120]}...")
    print()

Q: How did Norris feel about winning his first race?
2026-03-03 12:36:45 [info     ] retriever.query                query=How did Norris feel about winning his first race? top_k=2
2026-03-03 12:36:45 [info     ] embedder.encoding              batch_size=32 num_chunks=1


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.59s/it]

2026-03-03 12:36:46 [info     ] embedder.done                  num_embedded=1



Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


2026-03-03 12:36:46 [info     ] vector_store.query.done        num_hits=2 top_k=2
2026-03-03 12:36:46 [info     ] retriever.done                 num_results=2
  [0.461] singapore_gp — bit, you know, not always finishing behind, but we'll see how that goes. Q: And physically, we can see the sweat on your...
  [0.486] sao_paulo_gp — got by. And from there onwards, I just tried to look after the tyres, because you never know what was going to happen to...

Q: What was Verstappen's strategy in the wet conditions at Sao Paulo?
2026-03-03 12:36:46 [info     ] retriever.query                query=What was Verstappen's strategy in the wet conditions at Sao Paulo? top_k=2
2026-03-03 12:36:46 [info     ] embedder.encoding              batch_size=32 num_chunks=1


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.81it/s]

2026-03-03 12:36:46 [info     ] embedder.done                  num_embedded=1
2026-03-03 12:36:46 [info     ] vector_store.query.done        num_hits=2 top_k=2
2026-03-03 12:36:46 [info     ] retriever.done                 num_results=2
  [0.536] sao_paulo_gp — race. Did you look like it could have been on the podium since that moment? Esteban OCON: I mean, what a day that was, y...
  [0.571] sao_paulo_gp — strategic mistake we had. Now there was a lot more at stake. So I had to be more controlled, more aware of the champions...

Q: What did Hamilton say about his future at Mercedes?
2026-03-03 12:36:46 [info     ] retriever.query                query=What did Hamilton say about his future at Mercedes? top_k=2
2026-03-03 12:36:46 [info     ] embedder.encoding              batch_size=32 num_chunks=1



Batches: 100%|██████████| 1/1 [00:00<00:00, 99.36it/s]

2026-03-03 12:36:46 [info     ] embedder.done                  num_embedded=1


2026-03-03 12:36:46 [info     ] vector_store.query.done        num_hits=2 top_k=2
2026-03-03 12:36:46 [info     ] retriever.done                 num_results=2
  [0.483] abu_dhabi_gp — that I'm thinking about. Obviously, I'm trying to think about making sure that finish off the right way and the best way...
  [0.491] abu_dhabi_gp — but I was very touched when I saw this week again, a reward, a gala night, a reward which was given to Nelson Piquet. An...

